# Hackathon AI & ML — Advanced
**Base** : même pipeline que `baseline.ipynb`  
**Améliorations** :
- Seuil de NaN à 70% (vs 80%) → plus de features
- Features NaN par composante (labo / exam / questionnaire)
- `scale_pos_weight` pour le déséquilibre de classes
- **Meilleurs hyperparamètres LightGBM** issus d'Optuna (50 trials)
- Ensemble LightGBM + CatBoost + Random Forest (OOF)
- **Poids d'ensemble optimisés** par Nelder-Mead sur OOF
- **Stacking** LogisticRegression sur OOF
- **Threshold optimal** sur OOF
- Sélection automatique de la meilleure stratégie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

DATA_DIR          = Path('data')
GROUP_ID          = 'G1'
SUBMISSION_ID     = '4'
RANDOM_STATE      = 42
N_FOLDS           = 5
MISSING_THRESHOLD = 0.70   # vs 0.80 baseline → conserve plus de features

print('Librairies chargées OK')

## 1. Chargement des données

In [ ]:
print('Chargement des données...')
data     = pd.read_csv(DATA_DIR / 'data.csv')
labels   = pd.read_csv(DATA_DIR / 'ground_truth_train.csv')
test_idx = pd.read_csv(DATA_DIR / 'test_indexes.csv', header=None, names=['SEQN'])
metadata = pd.read_csv(DATA_DIR / 'features_metadata.csv')

print(f'data.csv        : {data.shape}')
print(f'ground_truth    : {labels.shape}')
print(f'test_indexes    : {test_idx.shape}')
print(f'features_meta   : {metadata.shape}')

# Split train / test
test_seqn  = set(test_idx['SEQN'].values)
train_seqn = set(labels['SEQN'].values) - test_seqn

train_data = data[data['SEQN'].isin(train_seqn)].copy()
test_data  = data[data['SEQN'].isin(test_seqn)].copy()
train_df   = train_data.merge(labels[labels['SEQN'].isin(train_seqn)], on='SEQN')

print(f'\nTrain : {train_df.shape}  |  Test : {test_data.shape}')
print(f'Taux de mortalité : {train_df["MORTSTAT_2019"].mean():.2%}')

## 2. Feature engineering — patterns de NaN

In [ ]:
# Récupérer les colonnes par composante (colonne 'SAS' dans features_metadata.csv)
meta_cols  = metadata[['SAS', 'Component']].rename(columns={'SAS': 'col'})
labo_cols  = meta_cols[meta_cols['Component'] == 'Laboratory']['col'].tolist()
exam_cols  = meta_cols[meta_cols['Component'] == 'Examination']['col'].tolist()
quest_cols = meta_cols[meta_cols['Component'] == 'Questionnaire']['col'].tolist()

def add_missing_features(df, labo_cols, exam_cols, quest_cols):
    all_cols = [c for c in df.columns if c not in ['SEQN', 'MORTSTAT_2019']]
    df = df.copy()
    df['feat_nb_missing_total'] = df[all_cols].isnull().sum(axis=1)
    df['feat_pct_missing_total'] = df['feat_nb_missing_total'] / len(all_cols)
    for name, cols in [('labo', labo_cols), ('exam', exam_cols), ('quest', quest_cols)]:
        present = [c for c in cols if c in df.columns]
        if present:
            df[f'feat_nb_missing_{name}'] = df[present].isnull().sum(axis=1)
    return df

train_df  = add_missing_features(train_df,  labo_cols, exam_cols, quest_cols)
test_data = add_missing_features(test_data, labo_cols, exam_cols, quest_cols)
print('Features NaN par composante ajoutées')

## 3. Nettoyage et préparation des features

In [ ]:
# Suppression colonnes > 70% manquant (vs 80% dans baseline)
missing_ratio = train_df.isnull().mean()
cols_to_drop  = [c for c in missing_ratio[missing_ratio > MISSING_THRESHOLD].index
                 if c not in ['SEQN', 'MORTSTAT_2019']]

train_clean = train_df.drop(columns=cols_to_drop)
test_clean  = test_data.drop(columns=[c for c in cols_to_drop if c in test_data.columns])

feature_cols    = [c for c in train_clean.columns if c not in ['SEQN', 'MORTSTAT_2019']]
X               = train_clean[feature_cols]
y               = train_clean['MORTSTAT_2019']
X_test          = test_clean[feature_cols]
test_seqn_order = test_clean['SEQN']

neg_pos_ratio = (y == 0).sum() / (y == 1).sum()

print(f'Colonnes supprimées (>{MISSING_THRESHOLD*100:.0f}% manquant) : {len(cols_to_drop)}')
print(f'Features finales    : {X.shape[1]}')
print(f'Exemples train      : {X.shape[0]}')
print(f'Exemples test       : {X_test.shape[0]}')
print(f'Déséquilibre (0/1)  : {neg_pos_ratio:.1f}x  → scale_pos_weight={neg_pos_ratio:.1f}')

## 4. Paramètres LightGBM — issus d'Optuna (50 trials)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Meilleurs hyperparamètres trouvés par Optuna (50 trials sur optuna.ipynb)
best_lgb_params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'verbose':           -1,
    'n_jobs':            -1,
    'random_state':      RANDOM_STATE,
    'n_estimators':      2000,
    # scale_pos_weight défini après chargement des données
    'learning_rate':     0.023688639503640783,
    'num_leaves':        244,
    'min_child_samples': 76,
    'subsample':         0.7993292420985183,
    'colsample_bytree':  0.5780093202212182,
    'reg_alpha':         0.004207053950287938,
    'reg_lambda':        0.0017073967431528124,
}

print('Paramètres LightGBM (Optuna) chargés')
for k, v in best_lgb_params.items():
    print(f'  {k}: {v}')

## 5. Entraînement CV — OOF predictions (LGB + CatBoost + RF)

In [ ]:
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
oof_rf   = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))
test_rf  = np.zeros(len(X_test))

# Imputation médiane pour RF (ne gère pas les NaN nativement)
train_medians = X.median()
X_rf      = X.fillna(train_medians)
X_test_rf = X_test.fillna(train_medians)

cat_params = {
    'iterations':           1000,
    'learning_rate':        0.05,
    'depth':                6,
    'loss_function':        'Logloss',
    'eval_metric':          'F1',
    'class_weights':        [1, neg_pos_ratio],
    'random_seed':          RANDOM_STATE,
    'verbose':              0,
    'early_stopping_rounds': 50,
}

rf_params = {
    'n_estimators':    500,
    'max_depth':       20,
    'min_samples_leaf': 10,
    'max_features':    'sqrt',
    'class_weight':    'balanced',
    'random_state':    RANDOM_STATE,
    'n_jobs':          -1,
}

print('Paramètres initialisés')
print(f'  CatBoost : {cat_params}')
print(f'  RF       : {rf_params}')

In [ ]:
# --- LightGBM (params Optuna) ---
print('=== LightGBM (params Optuna) ===')
lgb_params = {**best_lgb_params, 'scale_pos_weight': neg_pos_ratio}
f1_lgb = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(period=-1)])
    oof_lgb[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_lgb         += m.predict_proba(X_test)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_lgb[val_idx] > 0.5).astype(int))
    f1_lgb.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')
print(f'  LGB CV F1: {np.mean(f1_lgb):.4f} ± {np.std(f1_lgb):.4f}')

In [ ]:
# --- CatBoost ---
print('=== CatBoost ===')
f1_cat = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = CatBoostClassifier(**cat_params)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    oof_cat[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_cat         += m.predict_proba(X_test)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_cat[val_idx] > 0.5).astype(int))
    f1_cat.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')
print(f'  CAT CV F1: {np.mean(f1_cat):.4f} ± {np.std(f1_cat):.4f}')

In [ ]:
# --- Random Forest ---
print('=== Random Forest ===')
f1_rf = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_rf, y)):
    X_tr, X_val = X_rf.iloc[tr_idx], X_rf.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = RandomForestClassifier(**rf_params)
    m.fit(X_tr, y_tr)
    oof_rf[val_idx]  = m.predict_proba(X_val)[:, 1]
    test_rf         += m.predict_proba(X_test_rf)[:, 1] / N_FOLDS
    f1 = f1_score(y_val, (oof_rf[val_idx] > 0.5).astype(int))
    f1_rf.append(f1)
    print(f'  Fold {fold+1} F1: {f1:.4f}')
print(f'  RF  CV F1: {np.mean(f1_rf):.4f} ± {np.std(f1_rf):.4f}')

## 6. Ensemble — poids optimisés + stacking

In [ ]:
# --- A. Averaging avec poids optimisés par Nelder-Mead ---
def neg_f1_weighted(weights, threshold=0.5):
    w = np.abs(weights) / np.abs(weights).sum()
    oof = w[0]*oof_lgb + w[1]*oof_cat + w[2]*oof_rf
    return -f1_score(y, (oof > threshold).astype(int))

res = minimize(neg_f1_weighted, [1/3, 1/3, 1/3], method='Nelder-Mead',
               options={'maxiter': 1000, 'xatol': 1e-4})
opt_w = np.abs(res.x) / np.abs(res.x).sum()

oof_avg  = opt_w[0]*oof_lgb  + opt_w[1]*oof_cat  + opt_w[2]*oof_rf
test_avg = opt_w[0]*test_lgb + opt_w[1]*test_cat + opt_w[2]*test_rf

# Threshold optimal sur averaging
thresholds   = np.arange(0.20, 0.70, 0.01)
f1_avg_thr   = [f1_score(y, (oof_avg > t).astype(int)) for t in thresholds]
best_thr_avg = thresholds[np.argmax(f1_avg_thr)]
best_f1_avg  = max(f1_avg_thr)

print(f'Poids optimisés : LGB={opt_w[0]:.3f}  CAT={opt_w[1]:.3f}  RF={opt_w[2]:.3f}')
print(f'Averaging OOF F1 (thr={best_thr_avg:.2f}) : {best_f1_avg:.4f}')

In [ ]:
# --- B. Stacking — meta-learner LogisticRegression sur OOF ---
meta_X_train = np.column_stack([oof_lgb, oof_cat, oof_rf])
meta_X_test  = np.column_stack([test_lgb, test_cat, test_rf])

scaler = StandardScaler()
meta_X_train_s = scaler.fit_transform(meta_X_train)
meta_X_test_s  = scaler.transform(meta_X_test)

meta = LogisticRegression(C=1.0, random_state=RANDOM_STATE, max_iter=1000)
meta.fit(meta_X_train_s, y)

oof_stack  = meta.predict_proba(meta_X_train_s)[:, 1]
test_stack = meta.predict_proba(meta_X_test_s)[:, 1]

# Threshold optimal sur stacking
f1_stack_thr   = [f1_score(y, (oof_stack > t).astype(int)) for t in thresholds]
best_thr_stack = thresholds[np.argmax(f1_stack_thr)]
best_f1_stack  = max(f1_stack_thr)

print(f'Stacking OOF F1 (thr={best_thr_stack:.2f}) : {best_f1_stack:.4f}')

# --- Comparaison ---
print('\n=== Comparaison OOF F1 ===')
print(f'Baseline LGB (seuil 0.5)  : 0.6945')
print(f'Baseline LGB + seuil + spw: 0.7023')
print(f'LGB Optuna                : {np.mean(f1_lgb):.4f}')
print(f'CatBoost                  : {np.mean(f1_cat):.4f}')
print(f'Random Forest             : {np.mean(f1_rf):.4f}')
print(f'Averaging optimisé        : {best_f1_avg:.4f}  (thr={best_thr_avg:.2f})')
print(f'Stacking                  : {best_f1_stack:.4f}  (thr={best_thr_stack:.2f})')

In [ ]:
# Courbes threshold — Averaging vs Stacking
plt.figure(figsize=(9, 4))
plt.plot(thresholds, f1_avg_thr,   label=f'Averaging (best={best_f1_avg:.4f})')
plt.plot(thresholds, f1_stack_thr, label=f'Stacking  (best={best_f1_stack:.4f})')
plt.axvline(best_thr_avg,   color='C0', linestyle='--', alpha=0.6)
plt.axvline(best_thr_stack, color='C1', linestyle='--', alpha=0.6)
plt.xlabel('Threshold')
plt.ylabel('F1 OOF')
plt.title('F1 vs Threshold — Averaging vs Stacking')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Soumission — meilleure stratégie automatique

In [ ]:
# Sélection automatique de la meilleure stratégie
if best_f1_stack >= best_f1_avg:
    best_strategy  = 'Stacking'
    y_pred_proba   = test_stack
    best_threshold = best_thr_stack
    best_f1        = best_f1_stack
else:
    best_strategy  = 'Averaging'
    y_pred_proba   = test_avg
    best_threshold = best_thr_avg
    best_f1        = best_f1_avg

print(f'Stratégie choisie : {best_strategy} (F1 OOF={best_f1:.4f}, thr={best_threshold:.2f})')

y_pred_test = (y_pred_proba > best_threshold).astype(int)

submission = pd.DataFrame({
    'SEQN': test_seqn_order.values,
    'prediction': y_pred_test
}).sort_values('SEQN')

assert len(submission) == 5000, f'ERREUR : {len(submission)} lignes'

filename = f'{GROUP_ID}_{SUBMISSION_ID}.csv'
submission.to_csv(filename, index=False, header=False)

print(f'Fichier : {filename}')
print(f'Lignes  : {len(submission)}')
print(f'Répartition : {submission["prediction"].value_counts().to_dict()}')